# Kiểm thử độc lập các thuật toán phân loại tư thế trên video mới

Notebook này dùng dataset chính `posture_data_2fps.csv` để train các thuật toán phân loại tư thế từ landmark MediaPipe Pose.

Dataset `posture_external_test_2fps.csv` chỉ được dùng để test độc lập, không tham gia huấn luyện. External test giúp đánh giá khả năng tổng quát hóa trên video mới chưa từng xuất hiện trong quá trình train.

Bài toán sử dụng 99 đặc trưng landmark, tương ứng 33 điểm pose nhân với 3 tọa độ `(x, y, z)`. Nhãn `0` là đúng tư thế, nhãn `1` là sai tư thế. Recall của lớp sai tư thế là chỉ số quan trọng vì hệ thống cần hạn chế bỏ sót các trường hợp ngồi sai.

In [ ]:
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import ReduceLROnPlateau

OUTPUT_DIR = "/kaggle/working"
SEED = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
np.random.seed(SEED)
tf.random.set_seed(SEED)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 120)

print("TensorFlow version:", tf.__version__)
print("Output dir:", OUTPUT_DIR)

## 1. Tìm file input trong `/kaggle/input`

Dataset được Add Input vào Kaggle. Notebook sẽ tự tìm hai file CSV trong `/kaggle/input` để không phụ thuộc vào tên thư mục dataset trên Kaggle.

In [ ]:
def find_file(filename):
    """Tim filename trong /kaggle/input bang os.walk."""
    input_root = "/kaggle/input"
    for root, dirs, files in os.walk(input_root):
        if filename in files:
            return os.path.join(root, filename)
    raise FileNotFoundError(f"Khong tim thay {filename} trong {input_root}")


train_csv_path = find_file("posture_data_2fps.csv")
external_csv_path = find_file("posture_external_test_2fps.csv")

print("Train CSV:", train_csv_path)
print("External test CSV:", external_csv_path)

## 2. Load và kiểm tra dataset train

Dataset chính chỉ dùng để huấn luyện các thuật toán. External test không được dùng trong quá trình train. Ở notebook này, tập kiểm thử chính là external test từ video mới nên không chia thêm test nội bộ từ dataset chính.

In [ ]:
def validate_landmark_dataframe(df, dataset_name):
    """Kiem tra cau truc CSV landmark: 99 feature + label."""
    if "label" not in df.columns:
        raise ValueError(f"{dataset_name}: Khong co cot label")

    feature_columns = [col for col in df.columns if col != "label"]
    if len(feature_columns) != 99:
        raise ValueError(f"{dataset_name}: Can 99 feature, nhung hien co {len(feature_columns)} feature")

    labels = set(df["label"].dropna().unique())
    if not labels.issubset({0, 1, 0.0, 1.0}):
        raise ValueError(f"{dataset_name}: Label chi duoc gom 0 va 1, nhung hien co {sorted(labels)}")

    missing_rows = int(df.isna().any(axis=1).sum())
    if missing_rows > 0:
        print(f"{dataset_name}: Co {missing_rows} dong bi missing values. Tien hanh drop cac dong nay.")
        df = df.dropna().reset_index(drop=True)
    else:
        print(f"{dataset_name}: Khong co missing values.")

    df["label"] = df["label"].astype(int)
    return df


df_train_full = pd.read_csv(train_csv_path)

print("Shape train full:", df_train_full.shape)
display(df_train_full.head())
print("\nInfo train full:")
df_train_full.info()
print("\nPhan bo label train full:")
print(df_train_full["label"].value_counts().sort_index())
print("\nMissing values train full:")
print(df_train_full.isna().sum().sort_values(ascending=False).head(20))

df_train_full = validate_landmark_dataframe(df_train_full, "Train full")

X_full = df_train_full.drop("label", axis=1)
y_full = df_train_full["label"]

X_train, X_val, y_train, y_val = train_test_split(
    X_full,
    y_full,
    test_size=0.15,
    random_state=SEED,
    stratify=y_full
)

print("\nTrain shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("\nPhan bo label train:")
print(y_train.value_counts().sort_index())
print("\nPhan bo label validation:")
print(y_val.value_counts().sort_index())

## 3. Load và kiểm tra external test

Tập external test được tạo từ video mới không tham gia huấn luyện, dùng để đánh giá khả năng tổng quát hóa thực tế của các thuật toán.

In [ ]:
df_ext = pd.read_csv(external_csv_path)

print("Shape external test:", df_ext.shape)
display(df_ext.head())
print("\nInfo external test:")
df_ext.info()
print("\nPhan bo label external test:")
print(df_ext["label"].value_counts().sort_index())
print("\nMissing values external test:")
print(df_ext.isna().sum().sort_values(ascending=False).head(20))

df_ext = validate_landmark_dataframe(df_ext, "External test")

X_ext = df_ext.drop("label", axis=1)
y_ext = df_ext["label"]

missing_ext_columns = [col for col in X_full.columns if col not in X_ext.columns]
extra_ext_columns = [col for col in X_ext.columns if col not in X_full.columns]
if missing_ext_columns or extra_ext_columns:
    raise ValueError(
        "Cot feature cua external test khong khop voi train. "
        f"Missing: {missing_ext_columns}, Extra: {extra_ext_columns}"
    )

X_ext = X_ext[X_full.columns]

print("\nExternal test shape sau khi sap xep cot:", X_ext.shape)
print("Phan bo label external test sau kiem tra:")
print(y_ext.value_counts().sort_index())

## 4. Hàm đánh giá chung

Các thuật toán được đánh giá bằng Accuracy, Precision, Recall, F1-score và thời gian dự đoán. Trong đó, Recall của lớp `1 - sai tư thế` rất quan trọng vì đây là khả năng phát hiện các trường hợp cần cảnh báo.

In [ ]:
def evaluate_predictions(model_name, y_true, y_pred, train_time, predict_time):
    """Tinh cac metric chinh voi positive class = 1 - sai tu the."""
    sample_count = len(y_true)
    avg_predict_ms = (predict_time / sample_count) * 1000 if sample_count > 0 else 0.0

    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Train Time (s)": train_time,
        "Predict Time (s)": predict_time,
        "Avg Predict Time / Sample (ms)": avg_predict_ms,
    }


results = []
confusion_matrices = {}
classification_reports = {}
predictions = {}

## 5. Train và external test các thuật toán scikit-learn

Các thuật toán học máy truyền thống được train trên dataset chính và đánh giá trên external test. Các thuật toán cần chuẩn hóa dữ liệu sẽ dùng `Pipeline` với `StandardScaler` để scaler chỉ học từ tập train.

In [ ]:
sklearn_models = {
    "Logistic Regression": {
        "model": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000, random_state=SEED))
        ]),
        "filename": "logistic_regression_external.pkl",
    },
    "SVM": {
        "model": Pipeline([
            ("scaler", StandardScaler()),
            ("model", SVC(kernel="rbf", random_state=SEED))
        ]),
        "filename": "svm_external.pkl",
    },
    "KNN": {
        "model": Pipeline([
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier(n_neighbors=5))
        ]),
        "filename": "knn_external.pkl",
    },
    "Decision Tree": {
        "model": DecisionTreeClassifier(random_state=SEED),
        "filename": "decision_tree_external.pkl",
    },
    "Random Forest": {
        "model": RandomForestClassifier(
            n_estimators=200,
            random_state=SEED,
            n_jobs=-1
        ),
        "filename": "random_forest_external.pkl",
    },
    "Gradient Boosting": {
        "model": GradientBoostingClassifier(random_state=SEED),
        "filename": "gradient_boosting_external.pkl",
    },
}

for model_name, config in sklearn_models.items():
    print(f"\n===== {model_name} =====")
    model = config["model"]

    start = time.perf_counter()
    model.fit(X_train, y_train)
    train_time = time.perf_counter() - start

    start = time.perf_counter()
    y_pred = model.predict(X_ext)
    predict_time = time.perf_counter() - start

    metrics = evaluate_predictions(model_name, y_ext, y_pred, train_time, predict_time)
    results.append(metrics)

    confusion_matrices[model_name] = confusion_matrix(y_ext, y_pred, labels=[0, 1])
    classification_reports[model_name] = classification_report(
        y_ext,
        y_pred,
        labels=[0, 1],
        target_names=["0 - Correct", "1 - Incorrect"],
        zero_division=0
    )
    predictions[model_name] = y_pred

    model_path = os.path.join(OUTPUT_DIR, config["filename"])
    joblib.dump(model, model_path)

    print("Saved model:", model_path)
    print(metrics)
    print(classification_reports[model_name])

## 6. Train và external test ANN

ANN được train trên vector 99 đặc trưng landmark. `StandardScaler` chỉ được fit trên tập train, sau đó transform validation và external test để tránh rò rỉ dữ liệu từ tập kiểm thử độc lập.

In [ ]:
scaler_ann = StandardScaler()
X_train_scaled = scaler_ann.fit_transform(X_train)
X_val_scaled = scaler_ann.transform(X_val)
X_ext_scaled = scaler_ann.transform(X_ext)

ann_model = Sequential([
    Dense(128, activation="relu", input_shape=(99,)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dropout(0.25),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid"),
])

ann_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

ann_model.summary()

ann_best_path = os.path.join(OUTPUT_DIR, "ann_external_best_2fps.keras")
callbacks = [
    EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6),
    ModelCheckpoint(
        ann_best_path,
        save_best_only=True,
        monitor="val_loss",
        mode="min"
    ),
]

start = time.perf_counter()
history = ann_model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)
train_time_ann = time.perf_counter() - start

start = time.perf_counter()
y_prob_ann = ann_model.predict(X_ext_scaled)
predict_time_ann = time.perf_counter() - start

y_pred_ann = (y_prob_ann >= 0.5).astype(int).ravel()

ann_metrics = evaluate_predictions("ANN", y_ext, y_pred_ann, train_time_ann, predict_time_ann)
results.append(ann_metrics)

confusion_matrices["ANN"] = confusion_matrix(y_ext, y_pred_ann, labels=[0, 1])
classification_reports["ANN"] = classification_report(
    y_ext,
    y_pred_ann,
    labels=[0, 1],
    target_names=["0 - Correct", "1 - Incorrect"],
    zero_division=0
)
predictions["ANN"] = y_pred_ann

scaler_ann_path = os.path.join(OUTPUT_DIR, "scaler_ann_external_2fps.pkl")
joblib.dump(scaler_ann, scaler_ann_path)

print("Saved ANN best model:", ann_best_path)
print("Saved ANN scaler:", scaler_ann_path)
print(ann_metrics)
print(classification_reports["ANN"])

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Train loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("ANN Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
loss_plot_path = os.path.join(OUTPUT_DIR, "ann_training_loss_external_2fps.png")
plt.savefig(loss_plot_path, dpi=150)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Train accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.title("ANN Training Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
acc_plot_path = os.path.join(OUTPUT_DIR, "ann_training_accuracy_external_2fps.png")
plt.savefig(acc_plot_path, dpi=150)
plt.show()

print("Saved plot:", loss_plot_path)
print("Saved plot:", acc_plot_path)

## 7. Tạo bảng kết quả external test

Bảng này thể hiện kết quả thực tế hơn vì các thuật toán được đánh giá trên video mới không tham gia huấn luyện.

In [ ]:
result_columns = [
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "Train Time (s)",
    "Predict Time (s)",
    "Avg Predict Time / Sample (ms)",
]

external_results_df = pd.DataFrame(results)[result_columns]
external_results_df = external_results_df.sort_values(by="F1-score", ascending=False).reset_index(drop=True)

round_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "Train Time (s)",
    "Predict Time (s)",
    "Avg Predict Time / Sample (ms)",
]
external_results_df[round_columns] = external_results_df[round_columns].round(4)

display(external_results_df)

external_results_path = os.path.join(OUTPUT_DIR, "external_test_all_algorithms_2fps.csv")
external_results_df.to_csv(external_results_path, index=False, encoding="utf-8-sig")
print("Saved:", external_results_path)

## 8. So sánh internal test trước đó và external test

Kết quả internal test là kết quả chia ngẫu nhiên từ dataset chính. Kết quả external test là kết quả trên video mới, có ý nghĩa đánh giá khả năng tổng quát hóa thực tế hơn.

In [ ]:
internal_results = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Accuracy": 0.8948,
        "Precision": 0.9170,
        "Recall": 0.9059,
        "F1-score": 0.9114,
        "Avg Predict Time / Sample (ms)": 0.0040,
    },
    {
        "Model": "SVM",
        "Accuracy": 0.9940,
        "Precision": 0.9959,
        "Recall": 0.9939,
        "F1-score": 0.9949,
        "Avg Predict Time / Sample (ms)": 0.1191,
    },
    {
        "Model": "KNN",
        "Accuracy": 0.9933,
        "Precision": 0.9949,
        "Recall": 0.9939,
        "F1-score": 0.9944,
        "Avg Predict Time / Sample (ms)": 0.0749,
    },
    {
        "Model": "Decision Tree",
        "Accuracy": 0.9843,
        "Precision": 0.9898,
        "Recall": 0.9838,
        "F1-score": 0.9868,
        "Avg Predict Time / Sample (ms)": 0.0014,
    },
    {
        "Model": "Random Forest",
        "Accuracy": 0.9970,
        "Precision": 0.9980,
        "Recall": 0.9970,
        "F1-score": 0.9975,
        "Avg Predict Time / Sample (ms)": 0.0553,
    },
    {
        "Model": "Gradient Boosting",
        "Accuracy": 0.9933,
        "Precision": 0.9919,
        "Recall": 0.9970,
        "F1-score": 0.9944,
        "Avg Predict Time / Sample (ms)": 0.0041,
    },
    {
        "Model": "ANN",
        "Accuracy": 0.9970,
        "Precision": 0.9990,
        "Recall": 0.9960,
        "F1-score": 0.9975,
        "Avg Predict Time / Sample (ms)": 0.2046,
    },
])

compare_columns = [
    "Model",
    "Test Type",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "Avg Predict Time / Sample (ms)",
]

internal_compare = internal_results.copy()
internal_compare["Test Type"] = "Internal Test"
internal_compare = internal_compare[compare_columns]

external_compare = external_results_df[
    ["Model", "Accuracy", "Precision", "Recall", "F1-score", "Avg Predict Time / Sample (ms)"]
].copy()
external_compare["Test Type"] = "External Test"
external_compare = external_compare[compare_columns]

internal_vs_external_df = pd.concat([internal_compare, external_compare], ignore_index=True)
internal_vs_external_df = internal_vs_external_df.sort_values(["Model", "Test Type"]).reset_index(drop=True)

display(internal_vs_external_df)

internal_vs_external_path = os.path.join(OUTPUT_DIR, "internal_vs_external_all_algorithms_2fps.csv")
internal_vs_external_df.to_csv(internal_vs_external_path, index=False, encoding="utf-8-sig")
print("Saved:", internal_vs_external_path)

## 9. Vẽ biểu đồ external test

Các biểu đồ giúp quan sát trực quan thuật toán nào hoạt động tốt nhất trên tập video mới. Notebook dùng `matplotlib`, không dùng `seaborn`.

In [ ]:
def save_bar_chart(df, metric, title, ylabel, filename, rotate=True):
    plot_df = df.sort_values(metric, ascending=False).reset_index(drop=True)
    plt.figure(figsize=(10, 5))
    bars = plt.bar(plot_df["Model"], plot_df[metric], color="#4C78A8")
    plt.title(title)
    plt.xlabel("Thuật toán")
    plt.ylabel(ylabel)
    if rotate:
        plt.xticks(rotation=30, ha="right")
    plt.grid(axis="y", alpha=0.3)
    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            height,
            f"{height:.4f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )
    if metric != "Avg Predict Time / Sample (ms)":
        plt.ylim(0, min(1.05, max(1.0, plot_df[metric].max() + 0.05)))
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(path, dpi=150)
    plt.show()
    print("Saved:", path)


save_bar_chart(
    external_results_df,
    "Accuracy",
    "Accuracy trên external test",
    "Accuracy",
    "external_accuracy_all_algorithms_2fps.png",
)
save_bar_chart(
    external_results_df,
    "Precision",
    "Precision lớp sai tư thế trên external test",
    "Precision",
    "external_precision_all_algorithms_2fps.png",
)
save_bar_chart(
    external_results_df,
    "Recall",
    "Recall lớp sai tư thế trên external test",
    "Recall",
    "external_recall_all_algorithms_2fps.png",
)
save_bar_chart(
    external_results_df,
    "F1-score",
    "F1-score trên external test",
    "F1-score",
    "external_f1_all_algorithms_2fps.png",
)
save_bar_chart(
    external_results_df,
    "Avg Predict Time / Sample (ms)",
    "Thời gian dự đoán trung bình mỗi mẫu trên external test",
    "Milliseconds / sample",
    "external_predict_time_all_algorithms_2fps.png",
)

## 10. Vẽ biểu đồ internal vs external

So sánh internal test và external test giúp phát hiện mô hình có bị đánh giá quá cao khi chia ngẫu nhiên từ cùng dataset hay không.

In [ ]:
def save_internal_external_chart(metric, filename, title):
    model_order = internal_results["Model"].tolist()
    internal_values = internal_results.set_index("Model").loc[model_order, metric].values
    external_values = external_results_df.set_index("Model").loc[model_order, metric].values

    x = np.arange(len(model_order))
    width = 0.38

    plt.figure(figsize=(11, 5))
    plt.bar(x - width / 2, internal_values, width, label="Internal Test", color="#72B7B2")
    plt.bar(x + width / 2, external_values, width, label="External Test", color="#F58518")
    plt.title(title)
    plt.xlabel("Thuật toán")
    plt.ylabel(metric)
    plt.xticks(x, model_order, rotation=30, ha="right")
    plt.ylim(0, 1.05)
    plt.grid(axis="y", alpha=0.3)
    plt.legend()
    plt.tight_layout()

    path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(path, dpi=150)
    plt.show()
    print("Saved:", path)


save_internal_external_chart(
    "F1-score",
    "internal_vs_external_f1_all_algorithms_2fps.png",
    "So sánh F1-score: internal test và external test",
)
save_internal_external_chart(
    "Recall",
    "internal_vs_external_recall_all_algorithms_2fps.png",
    "So sánh Recall lớp sai tư thế: internal test và external test",
)
save_internal_external_chart(
    "Accuracy",
    "internal_vs_external_accuracy_all_algorithms_2fps.png",
    "So sánh Accuracy: internal test và external test",
)

## 11. Confusion Matrix external test

Ma trận nhầm lẫn cho biết số mẫu đúng/sai bị dự đoán nhầm. Với bài toán này, FN là trường hợp sai tư thế nhưng mô hình dự đoán đúng tư thế, cần hạn chế.

In [ ]:
confusion_matrix_filenames = {
    "Logistic Regression": "confusion_matrix_external_logistic_regression_2fps.png",
    "SVM": "confusion_matrix_external_svm_2fps.png",
    "KNN": "confusion_matrix_external_knn_2fps.png",
    "Decision Tree": "confusion_matrix_external_decision_tree_2fps.png",
    "Random Forest": "confusion_matrix_external_random_forest_2fps.png",
    "Gradient Boosting": "confusion_matrix_external_gradient_boosting_2fps.png",
    "ANN": "confusion_matrix_external_ann_2fps.png",
}


def save_confusion_matrix_plot(model_name, cm, filename):
    labels = ["0 - Correct", "1 - Incorrect"]
    plt.figure(figsize=(5, 4))
    plt.imshow(cm, interpolation="nearest", cmap="Blues")
    plt.title(f"Confusion Matrix - {model_name}")
    plt.colorbar()
    tick_marks = np.arange(len(labels))
    plt.xticks(tick_marks, labels, rotation=20, ha="right")
    plt.yticks(tick_marks, labels)
    plt.xlabel("Predicted label")
    plt.ylabel("True label")

    threshold = cm.max() / 2 if cm.max() > 0 else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            color = "white" if cm[i, j] > threshold else "black"
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", color=color, fontsize=12)

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(path, dpi=150)
    plt.show()
    print("Saved:", path)


for model_name, filename in confusion_matrix_filenames.items():
    save_confusion_matrix_plot(model_name, confusion_matrices[model_name], filename)

## 12. Classification report

Các classification report của external test được lưu chung vào một file text để tiện trích dẫn và đối chiếu trong báo cáo.

In [ ]:
reports_path = os.path.join(OUTPUT_DIR, "external_classification_reports_all_algorithms_2fps.txt")

with open(reports_path, "w", encoding="utf-8") as f:
    f.write("CLASSIFICATION REPORTS - EXTERNAL TEST - ALL ALGORITHMS - 2FPS\n")
    f.write("Positive class = 1 - Incorrect posture\n")
    f.write("=" * 80 + "\n\n")
    for model_name in [
        "Logistic Regression",
        "SVM",
        "KNN",
        "Decision Tree",
        "Random Forest",
        "Gradient Boosting",
        "ANN",
    ]:
        f.write(f"{model_name}\n")
        f.write("-" * len(model_name) + "\n")
        f.write(classification_reports[model_name])
        f.write("\n\n")

print("Saved:", reports_path)

## 13. Kết luận tự động

Kết luận tự động giúp lấy nhanh đoạn nhận xét ban đầu cho báo cáo. Khi viết luận văn, vẫn nên đọc lại confusion matrix và classification report để phân tích sâu hơn, đặc biệt là Recall của lớp sai tư thế.

In [ ]:
best_external = external_results_df.sort_values("F1-score", ascending=False).iloc[0]
best_internal = internal_results.sort_values("F1-score", ascending=False).iloc[0]

conclusion = (
    f"Trên tập kiểm thử độc lập được trích xuất từ các video mới không tham gia huấn luyện, "
    f"mô hình {best_external['Model']} đạt F1-score cao nhất là {best_external['F1-score']:.4f}, "
    f"Accuracy là {best_external['Accuracy']:.4f}, Recall lớp sai tư thế là {best_external['Recall']:.4f}. "
    "So với kết quả test nội bộ, hiệu quả của các thuật toán có thể giảm do external test chứa video mới, "
    "góc quay/người dùng/ánh sáng khác với dữ liệu huấn luyện. Đây là đánh giá thực tế hơn về khả năng "
    "tổng quát hóa của mô hình."
)

print(conclusion)

if best_external["Model"] != best_internal["Model"]:
    print(
        f"Thuật toán tốt nhất trên internal test là {best_internal['Model']}, "
        f"trong khi trên external test là {best_external['Model']}. "
        "Điều này cho thấy cần xem xét khả năng tổng quát hóa chứ không chỉ dựa trên kết quả chia ngẫu nhiên từ cùng dataset."
    )
else:
    print(
        f"Thuật toán tốt nhất trên internal test và external test đều là {best_external['Model']}. "
        "Tuy nhiên, external test vẫn là căn cứ quan trọng hơn để đánh giá khả năng hoạt động trên video mới."
    )

## 14. Danh sách file output

Cell cuối in các file chính được tạo trong `/kaggle/working` để thuận tiện tải về sau khi chạy notebook.

In [ ]:
expected_outputs = [
    "external_test_all_algorithms_2fps.csv",
    "internal_vs_external_all_algorithms_2fps.csv",
    "external_classification_reports_all_algorithms_2fps.txt",
    "external_accuracy_all_algorithms_2fps.png",
    "external_precision_all_algorithms_2fps.png",
    "external_recall_all_algorithms_2fps.png",
    "external_f1_all_algorithms_2fps.png",
    "external_predict_time_all_algorithms_2fps.png",
    "internal_vs_external_f1_all_algorithms_2fps.png",
    "internal_vs_external_recall_all_algorithms_2fps.png",
    "internal_vs_external_accuracy_all_algorithms_2fps.png",
    "confusion_matrix_external_logistic_regression_2fps.png",
    "confusion_matrix_external_svm_2fps.png",
    "confusion_matrix_external_knn_2fps.png",
    "confusion_matrix_external_decision_tree_2fps.png",
    "confusion_matrix_external_random_forest_2fps.png",
    "confusion_matrix_external_gradient_boosting_2fps.png",
    "confusion_matrix_external_ann_2fps.png",
    "logistic_regression_external.pkl",
    "svm_external.pkl",
    "knn_external.pkl",
    "decision_tree_external.pkl",
    "random_forest_external.pkl",
    "gradient_boosting_external.pkl",
    "ann_external_best_2fps.keras",
    "scaler_ann_external_2fps.pkl",
    "ann_training_loss_external_2fps.png",
    "ann_training_accuracy_external_2fps.png",
]

print("Danh sach file output trong /kaggle/working:")
for filename in expected_outputs:
    path = os.path.join(OUTPUT_DIR, filename)
    status = "OK" if os.path.exists(path) else "CHUA TAO"
    print(f"[{status}] {path}")

print("\nTat ca file hien co trong /kaggle/working:")
for filename in sorted(os.listdir(OUTPUT_DIR)):
    print(os.path.join(OUTPUT_DIR, filename))